# Getting Started with AnnData

**Assignment:** Special Topics — scRNA-seq Analysis, Part 2  
**References:**
- [anndata.readthedocs.io — Getting Started](https://anndata.readthedocs.io/en/latest/tutorials/notebooks/getting-started.html)
- [scverse-tutorials — AnnData Getting Started](https://scverse-tutorials.readthedocs.io/en/latest/notebooks/anndata_getting_started.html)

---

## What is AnnData?

`AnnData` (Annotated Data) is the core data structure used across the scverse ecosystem (Scanpy, scVI, etc.). It is purpose-built for matrix-like data where:
- Each **row** is an observation (e.g. a cell)
- Each **column** is a variable/feature (e.g. a gene)

The key design principle is that both axes are **indexed** and can carry rich metadata alongside the main data matrix. This makes it far more expressive than a plain NumPy array or a Pandas DataFrame alone.

### AnnData Structure at a Glance

```
AnnData object
│
├── .X          → Main data matrix (n_obs × n_vars), usually count or expression values
├── .obs        → Observation-level metadata DataFrame (n_obs × any) — e.g. cell type, batch
├── .var        → Variable-level metadata DataFrame (n_vars × any) — e.g. gene name, chromosome
├── .obsm       → Observation-level matrices — e.g. PCA embeddings, UMAP coordinates
├── .varm       → Variable-level matrices — e.g. PCA loadings
├── .obsp       → Pairwise observation matrices — e.g. connectivities graph
├── .uns        → Unstructured metadata dictionary — e.g. colour palettes, run parameters
└── .layers     → Alternative data matrices (same shape as .X) — e.g. raw counts, normalised
```


## 0. Installation and Imports

In [3]:
# Uncomment if running for the first time
!pip install anndata numpy pandas scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.3/174.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 81.6 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
import anndata as ad
from scipy.sparse import csr_matrix

print(f"anndata version: {ad.__version__}")
print(f"numpy version:   {np.__version__}")
print(f"pandas version:  {pd.__version__}")

anndata version: 0.12.11
numpy version:   2.0.2
pandas version:  2.2.2


/tmp/ipykernel_8635/1257162930.py:6: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(f"anndata version: {ad.__version__}")


---
## 1. Initializing an AnnData Object

The simplest AnnData object requires only the main data matrix `.X`. In scRNA-seq this is typically a **sparse count matrix** where rows are cells and columns are genes.

We use `scipy.sparse.csr_matrix` (Compressed Sparse Row) because scRNA-seq matrices are highly sparse — most genes are not expressed in most cells. Storing them as sparse matrices saves significant memory.

In [5]:
# Simulate a count matrix: 100 cells x 2000 genes
# np.random.poisson(1) gives low integer counts, mimicking real scRNA-seq sparsity
np.random.seed(42)
counts = csr_matrix(np.random.poisson(1, size=(100, 2000)), dtype=np.float32)

adata = ad.AnnData(counts)
adata

AnnData object with n_obs × n_vars = 100 × 2000

In [6]:
# .X holds the main matrix
print("Type of .X:", type(adata.X))
print("Shape:", adata.X.shape)
print("Stored (non-zero) elements:", adata.X.nnz)
print("Sparsity: {:.1%} of values are zero".format(1 - adata.X.nnz / (adata.n_obs * adata.n_vars)))

Type of .X: <class 'scipy.sparse._csr.csr_matrix'>
Shape: (100, 2000)
Stored (non-zero) elements: 126377
Sparsity: 36.8% of values are zero


### Naming Observations and Variables

By default, obs and var are given integer indices. In practice we always set meaningful names — barcodes for cells, gene IDs for genes.

In [7]:
# Assign cell barcodes (obs_names) and gene IDs (var_names)
adata.obs_names = [f"Cell_{i:03d}" for i in range(adata.n_obs)]
adata.var_names = [f"Gene_{i:04d}" for i in range(adata.n_vars)]

print("First 5 cell names:", adata.obs_names[:5].tolist())
print("First 5 gene names:", adata.var_names[:5].tolist())
print("\nSummary:", adata)

First 5 cell names: ['Cell_000', 'Cell_001', 'Cell_002', 'Cell_003', 'Cell_004']
First 5 gene names: ['Gene_0000', 'Gene_0001', 'Gene_0002', 'Gene_0003', 'Gene_0004']

Summary: AnnData object with n_obs × n_vars = 100 × 2000


### Subsetting AnnData

AnnData supports Pandas-style indexing. Subsetting returns a **view** (not a copy) by default — memory-efficient but read-only. Use `.copy()` to get an editable copy.

In [8]:
# Subset by name
subset_by_name = adata[["Cell_001", "Cell_010"], ["Gene_0005", "Gene_1999"]]
print("Subset by name:", subset_by_name)

# Subset by integer index
subset_by_idx = adata[:10, :50]  # first 10 cells, first 50 genes
print("Subset by index:", subset_by_idx)

# Check it is a view
print("Is view?", subset_by_idx.is_view)

Subset by name: View of AnnData object with n_obs × n_vars = 2 × 2
Subset by index: View of AnnData object with n_obs × n_vars = 10 × 50
Is view? True


In [9]:
# Making a copy to get an independent, writable object
adata_copy = adata[:10, :50].copy()
print("Is view?", adata_copy.is_view)  # False — it's an independent object

Is view? False


---
## 2. Adding Aligned Metadata: `obs` and `var`

`.obs` and `.var` are **Pandas DataFrames** indexed by `obs_names` and `var_names` respectively. Any column added here is automatically aligned to the correct axis.

### Observation-level metadata (`obs`)

Typical `obs` columns in scRNA-seq: cell type, batch, donor ID, total counts, % mitochondrial reads.

In [10]:
# Simulate cell type labels
cell_types = np.random.choice(["B cell", "T cell", "Monocyte", "NK cell"], size=adata.n_obs)
# Use pd.Categorical — more memory-efficient than plain strings for repeated values
adata.obs["cell_type"] = pd.Categorical(cell_types)

# Simulate batch labels (e.g. two sequencing runs)
adata.obs["batch"] = pd.Categorical(np.random.choice(["batch_1", "batch_2"], size=adata.n_obs))

# Simulate a continuous QC metric: total UMI counts per cell
adata.obs["total_counts"] = np.array(adata.X.sum(axis=1)).flatten()

adata.obs.head()

,cell_type,batch,total_counts
Cell_000,T cell,batch_1,1950.0
Cell_001,T cell,batch_1,1949.0
Cell_002,Monocyte,batch_2,1968.0
Cell_003,T cell,batch_1,1963.0
Cell_004,Monocyte,batch_2,2115.0


In [11]:
# The AnnData repr now reports obs columns
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type', 'batch', 'total_counts'

### Variable-level metadata (`var`)

Typical `var` columns: gene symbol, chromosome location, whether the gene is mitochondrial.

In [12]:
# Simulate whether each gene is mitochondrial (prefix MT- in real data)
adata.var["mt"] = np.random.choice([True, False], size=adata.n_vars, p=[0.02, 0.98])

# Simulate mean expression per gene
adata.var["mean_counts"] = np.array(adata.X.mean(axis=0)).flatten()

# Simulate a chromosome annotation
chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
adata.var["chromosome"] = pd.Categorical(np.random.choice(chroms, size=adata.n_vars))

adata.var.head()

,mt,mean_counts,chromosome
Gene_0000,False,0.830000,chr6
Gene_0001,False,1.000000,chr13
Gene_0002,False,1.079999,chr12
Gene_0003,False,1.140000,chr8
Gene_0004,False,0.970000,chr9


### Subsetting Using Metadata (Boolean Masking)

Because `obs` and `var` are DataFrames, we can use boolean expressions directly as subset indices.

In [13]:
# Keep only T cells
t_cells = adata[adata.obs["cell_type"] == "T cell"]
print("T cells only:", t_cells)

# Keep only non-mitochondrial genes
non_mt = adata[:, ~adata.var["mt"]]
print("Non-mitochondrial genes only:", non_mt)

# Combined filter: T cells AND non-mitochondrial genes
filtered = adata[adata.obs["cell_type"] == "T cell", ~adata.var["mt"]]
print("Combined filter:", filtered)

T cells only: View of AnnData object with n_obs × n_vars = 29 × 2000
    obs: 'cell_type', 'batch', 'total_counts'
    var: 'mt', 'mean_counts', 'chromosome'
Non-mitochondrial genes only: View of AnnData object with n_obs × n_vars = 100 × 1963
    obs: 'cell_type', 'batch', 'total_counts'
    var: 'mt', 'mean_counts', 'chromosome'
Combined filter: View of AnnData object with n_obs × n_vars = 29 × 1963
    obs: 'cell_type', 'batch', 'total_counts'
    var: 'mt', 'mean_counts', 'chromosome'


---
## 3. Observation and Variable Level Matrices: `obsm` and `varm`

`.obsm` and `.varm` store **multi-dimensional arrays** aligned to the observation or variable axis. The classic use cases are:

- `adata.obsm["X_pca"]` — PCA embedding of cells (shape: n_obs × n_components)
- `adata.obsm["X_umap"]` — UMAP 2D coordinates (shape: n_obs × 2)
- `adata.varm["PCs"]` — PCA loadings per gene (shape: n_vars × n_components)

Keys prefixed with `X_` are treated as embeddings by Scanpy's plotting functions.

In [14]:
# Simulate PCA result: 50 principal components per cell
adata.obsm["X_pca"] = np.random.normal(size=(adata.n_obs, 50))

# Simulate UMAP 2D coordinates
adata.obsm["X_umap"] = np.random.normal(size=(adata.n_obs, 2))

print("obsm keys:", list(adata.obsm.keys()))
print("PCA shape:", adata.obsm["X_pca"].shape)   # (100, 50)
print("UMAP shape:", adata.obsm["X_umap"].shape)  # (100, 2)

obsm keys: ['X_pca', 'X_umap']
PCA shape: (100, 50)
UMAP shape: (100, 2)


In [15]:
# Simulate PCA loadings per gene (varm)
# Shape must be (n_vars, n_components) — aligned to the variable axis
adata.varm["PCs"] = np.random.normal(size=(adata.n_vars, 50))

print("varm keys:", list(adata.varm.keys()))
print("PC loadings shape:", adata.varm["PCs"].shape)  # (2000, 50)

# obsm/varm arrays are accessible like a dictionary
print("\nFirst 3 cells, first 5 PCs:")
print(adata.obsm["X_pca"][:3, :5])

varm keys: ['PCs']
PC loadings shape: (2000, 50)

First 3 cells, first 5 PCs:
[[-0.55687631 -1.16992402 -0.29871932  1.56135297 -1.4792182 ]
 [-1.08984971  0.79491398  1.64279895  1.09717449  0.57024396]
 [-0.13949263  1.37025124  1.91768922 -0.36090521 -0.72056378]]


### Pairwise Observation Matrices: `obsp`

`.obsp` stores pairwise matrices between observations — most commonly the **k-nearest-neighbours graph** used for clustering (shape: n_obs × n_obs, sparse).

In [16]:
from scipy.sparse import random as sparse_random

# Simulate a sparse connectivity matrix (adjacency graph)
connectivity = sparse_random(adata.n_obs, adata.n_obs, density=0.05, format="csr", dtype=np.float32)
adata.obsp["connectivities"] = connectivity

print("obsp keys:", list(adata.obsp.keys()))
print("Connectivity matrix shape:", adata.obsp["connectivities"].shape)

obsp keys: ['connectivities']
Connectivity matrix shape: (100, 100)


---
## 4. Unstructured Metadata: `uns`

`.uns` is a plain Python **dictionary** for anything that doesn't fit the aligned structure above. Common uses:
- Colour palettes for cell types
- Tool parameters / run configuration
- Clustering result labels
- Dataset provenance

In [17]:
# Store a colour palette for cell types (used by sc.pl.umap)
adata.uns["cell_type_colors"] = {
    "B cell":   "#1f77b4",
    "T cell":   "#ff7f0e",
    "Monocyte": "#2ca02c",
    "NK cell":  "#d62728",
}

# Store dataset-level provenance
adata.uns["dataset_info"] = {
    "source": "simulated",
    "n_cells_original": 100,
    "organism": "Homo sapiens",
    "chemistry": "10X Chromium v3",
}

# Store the neighbours parameters (as Scanpy would)
adata.uns["neighbors"] = {
    "params": {"n_neighbors": 15, "metric": "euclidean"},
    "connectivities_key": "connectivities",
}

print("uns keys:", list(adata.uns.keys()))
print("\ndataset_info:", adata.uns["dataset_info"])

uns keys: ['cell_type_colors', 'dataset_info', 'neighbors']

dataset_info: {'source': 'simulated', 'n_cells_original': 100, 'organism': 'Homo sapiens', 'chemistry': '10X Chromium v3'}


In [18]:
# uns can store arbitrarily nested data — numpy arrays, lists, dicts, etc.
adata.uns["pca"] = {
    "variance_ratio": np.random.dirichlet(np.ones(50)),  # simulated explained variance
    "params": {"n_comps": 50, "use_highly_variable": True},
}

print("PCA variance ratio (first 5 PCs):", adata.uns["pca"]["variance_ratio"][:5])

PCA variance ratio (first 5 PCs): [0.00154766 0.00139775 0.00631475 0.02797218 0.01060431]


---
## 5. Layers

`.layers` stores **alternative matrices of the same shape** as `.X`. This is useful when you want to keep multiple representations of the data simultaneously, e.g.:

| Layer name | Typical content |
|---|---|
| `raw_counts` | Original integer UMI counts before any processing |
| `normalised` | Library-size normalised values |
| `log1p` | Log-transformed normalised values |
| `scaled` | Zero-mean, unit-variance scaled values |

The standard Scanpy workflow overwrites `.X` at each step but saves the originals in layers.

In [21]:
# Save the raw integer counts as a layer before any normalisation
adata.layers["raw_counts"] = adata.X.copy()

# Simulate library-size normalisation: scale each cell to 10,000 total counts
total_counts = np.array(adata.X.sum(axis=1)).flatten()  # shape (n_obs,)
# Perform element-wise multiplication, broadcasting total_counts to each row
# Ensure the result is explicitly a CSR matrix
normalised = adata.X.multiply(1e4 / total_counts[:, None]).tocsr() # explicitly convert to CSR

adata.layers["normalised"] = normalised

# Simulate log1p transformation (log(x+1)) — standard in scRNA-seq
# Ensure the copy is also a CSR matrix for consistency and to avoid potential issues
adata.layers["log1p"] = normalised.copy().tocsr()
adata.layers["log1p"].data = np.log1p(adata.layers["log1p"].data)

print("Layers:", list(adata.layers.keys()))
print("\nraw_counts[0, :5]:", adata.layers["raw_counts"][0, :5].toarray().flatten())
print("normalised[0, :5]: ", adata.layers["normalised"][0, :5].toarray().flatten().round(2))
print("log1p[0, :5]:      ", adata.layers["log1p"][0, :5].toarray().flatten().round(3))

Layers: ['raw_counts', 'normalised', 'log1p']

raw_counts[0, :5]: [1. 2. 0. 0. 3.]
normalised[0, :5]:  [ 5.13 10.26  0.    0.   15.38]
log1p[0, :5]:       [1.813 2.421 0.    0.    2.796]


In [22]:
# Layers are accessible and subsettable exactly like .X
# Here we look at layer values for the first 3 cells, first 5 genes
print("Subset from log1p layer (3 cells x 5 genes):")
print(adata[:3, :5].layers["log1p"].toarray())

Subset from log1p layer (3 cells x 5 genes):
[[1.812902  2.4209378 0.        0.        2.7963428]
 [0.        2.4214053 0.        2.4214053 0.       ]
 [2.412569  0.        1.8052186 1.8052186 1.8052186]]


In [23]:
# Full AnnData state after all additions
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type', 'batch', 'total_counts'
    var: 'mt', 'mean_counts', 'chromosome'
    uns: 'cell_type_colors', 'dataset_info', 'neighbors', 'pca'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw_counts', 'normalised', 'log1p'
    obsp: 'connectivities'

---
## 6. Conversion to DataFrames

`adata.to_df()` converts `.X` (or a specified layer) to a dense Pandas DataFrame with `obs_names` as the index and `var_names` as columns. Useful for exporting data or using standard Pandas operations.

> ⚠️ Be careful with large datasets: converting a sparse matrix to a dense DataFrame loads everything into memory.

In [24]:
# Convert .X to a DataFrame
df_X = adata.to_df()
print("Type:", type(df_X))
print("Shape:", df_X.shape)
df_X.iloc[:5, :6]  # first 5 cells, first 6 genes

Type: <class 'pandas.core.frame.DataFrame'>
Shape: (100, 2000)


,Gene_0000,Gene_0001,Gene_0002,Gene_0003,Gene_0004,Gene_0005
Cell_000,1.0,2.0,0.0,0.0,3.0,2.0
Cell_001,0.0,2.0,0.0,2.0,0.0,0.0
Cell_002,2.0,0.0,1.0,1.0,1.0,2.0
Cell_003,0.0,3.0,0.0,4.0,2.0,0.0
Cell_004,1.0,1.0,3.0,4.0,0.0,0.0


In [25]:
# Convert a specific layer to a DataFrame
df_log1p = adata.to_df(layer="log1p")
print("log1p layer as DataFrame:")
df_log1p.iloc[:5, :6].round(3)

log1p layer as DataFrame:


,Gene_0000,Gene_0001,Gene_0002,Gene_0003,Gene_0004,Gene_0005
Cell_000,1.813,2.421,0.000,0.000,2.796,2.421
Cell_001,0.000,2.421,0.000,2.421,0.000,0.000
Cell_002,2.413,0.000,1.805,1.805,1.805,2.413
Cell_003,0.000,2.790,0.000,3.062,2.415,0.000
Cell_004,1.745,1.745,2.720,2.991,0.000,0.000


In [26]:
# obs and var are already DataFrames — no conversion needed
print("adata.obs is a:", type(adata.obs))
print("\nCell type distribution:")
print(adata.obs["cell_type"].value_counts())

print("\nMitochondrial gene count:", adata.var["mt"].sum())

adata.obs is a: <class 'pandas.core.frame.DataFrame'>

Cell type distribution:
cell_type
T cell      29
Monocyte    25
NK cell     24
B cell      22
Name: count, dtype: int64

Mitochondrial gene count: 37


In [27]:
# You can merge obs metadata with expression values using standard pandas
# Example: mean expression per cell type for a single gene
gene_of_interest = "Gene_0001"
expr_series = df_log1p[gene_of_interest]
merged = adata.obs[["cell_type"]].copy()
merged["expression"] = expr_series.values
print(f"Mean {gene_of_interest} expression by cell type:")
print(merged.groupby("cell_type")["expression"].mean().round(4))

Mean Gene_0001 expression by cell type:
cell_type
B cell      1.2959
Monocyte    1.1802
NK cell     1.5582
T cell      1.0462
Name: expression, dtype: float32


/tmp/ipykernel_8635/870809703.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(merged.groupby("cell_type")["expression"].mean().round(4))


---
## 7. Writing and Reading `.h5ad` Files

The native AnnData format is `.h5ad` — an HDF5 file that stores the entire AnnData object (`.X`, `.obs`, `.var`, `.obsm`, `.varm`, `.uns`, `.layers`, `.obsp`) in a single file. It is compact, fast, and language-interoperable (can be read in R with `zellkonverter`).

Alternative formats: `.zarr` (cloud-friendly), `.loom`, CSV exports.

In [28]:
import os

output_path = "adata_getting_started.h5ad"

# Write to disk — saves ALL components (.X, .obs, .var, .obsm, .varm, .uns, .layers, .obsp)
adata.write_h5ad(output_path)

file_size_kb = os.path.getsize(output_path) / 1024
print(f"Written to: {output_path}")
print(f"File size:  {file_size_kb:.1f} KB")

Written to: adata_getting_started.h5ad
File size:  4969.0 KB


In [29]:
# Reading back from disk
adata_loaded = ad.read_h5ad(output_path)
adata_loaded

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type', 'batch', 'total_counts'
    var: 'mt', 'mean_counts', 'chromosome'
    uns: 'cell_type_colors', 'dataset_info', 'neighbors', 'pca'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'log1p', 'normalised', 'raw_counts'
    obsp: 'connectivities'

In [30]:
# Verify all components survived the round-trip
print("obs columns:  ", list(adata_loaded.obs.columns))
print("var columns:  ", list(adata_loaded.var.columns))
print("obsm keys:    ", list(adata_loaded.obsm.keys()))
print("varm keys:    ", list(adata_loaded.varm.keys()))
print("layers keys:  ", list(adata_loaded.layers.keys()))
print("uns keys:     ", list(adata_loaded.uns.keys()))
print("obsp keys:    ", list(adata_loaded.obsp.keys()))

obs columns:   ['cell_type', 'batch', 'total_counts']
var columns:   ['mt', 'mean_counts', 'chromosome']
obsm keys:     ['X_pca', 'X_umap']
varm keys:     ['PCs']
layers keys:   ['log1p', 'normalised', 'raw_counts']
uns keys:      ['cell_type_colors', 'dataset_info', 'neighbors', 'pca']
obsp keys:     ['connectivities']


In [31]:
# Confirm matrix values are identical
import numpy.testing as npt

npt.assert_array_equal(
    adata.X.toarray(),
    adata_loaded.X.toarray()
)
print("✓ .X values identical after round-trip")

npt.assert_array_almost_equal(
    adata.obsm["X_pca"],
    adata_loaded.obsm["X_pca"]
)
print("✓ .obsm['X_pca'] identical after round-trip")

✓ .X values identical after round-trip
✓ .obsm['X_pca'] identical after round-trip


In [32]:
# Writing with compression (reduces file size significantly for large datasets)
compressed_path = "adata_compressed.h5ad"
adata.write_h5ad(compressed_path, compression="gzip")

size_uncompressed = os.path.getsize(output_path) / 1024
size_compressed   = os.path.getsize(compressed_path) / 1024
print(f"Uncompressed: {size_uncompressed:.1f} KB")
print(f"Compressed:   {size_compressed:.1f} KB")
print(f"Reduction:    {100*(1 - size_compressed/size_uncompressed):.1f}%")

Uncompressed: 4969.0 KB
Compressed:   1906.3 KB
Reduction:    61.6%


---
## 8. Partial Reading of Large Data

For very large datasets (millions of cells, tens of thousands of genes), loading the entire `.h5ad` file into memory is impractical. AnnData supports **backed mode** — the `.X` matrix stays on disk and is only read on demand.

This is critical for production workflows where a full dataset may be 10–100 GB.

In [33]:
# First, let's create a slightly larger "large" dataset to demonstrate this
large_counts = csr_matrix(np.random.poisson(1, size=(500, 5000)), dtype=np.float32)
adata_large = ad.AnnData(large_counts)
adata_large.obs_names = [f"Cell_{i:04d}" for i in range(500)]
adata_large.var_names = [f"Gene_{i:05d}" for i in range(5000)]
adata_large.obs["cell_type"] = pd.Categorical(
    np.random.choice(["B cell", "T cell", "Monocyte"], size=500)
)
adata_large.write_h5ad("adata_large.h5ad")
print(f"Large dataset written: {os.path.getsize('adata_large.h5ad')/1024:.1f} KB")

Large dataset written: 12663.1 KB


In [34]:
# Backed mode: file stays on disk, metadata (.obs, .var, .uns) loaded into memory
# .X is NOT loaded — it is accessed via memory-mapped file I/O on demand
adata_backed = ad.read_h5ad("adata_large.h5ad", backed="r")  # 'r' = read-only

print("isbacked:", adata_backed.isbacked)    # True
print("filename:", adata_backed.filename)    # path to the .h5ad file
print("Shape:", adata_backed.shape)
print("Type of .X:", type(adata_backed.X))  # h5py Dataset, not an array in RAM

isbacked: True
filename: adata_large.h5ad
Shape: (500, 5000)
Type of .X: <class 'anndata._core.sparse_dataset._CSRDataset'>


In [35]:
# Slicing a backed AnnData only reads that slice from disk
# This is the key advantage: no need to load 100 GB to inspect 100 cells
small_slice = adata_backed[:10, :20]  # reads only rows 0-9, cols 0-19
print("Slice type:", type(small_slice))
print("Slice shape:", small_slice.shape)

# Convert the small slice to a dense array (now it's in memory)
print("\nFirst 3 cells, first 5 genes (from disk):")
print(small_slice.X[:3, :5])

Slice type: <class 'anndata._core.anndata.AnnData'>
Slice shape: (10, 20)

First 3 cells, first 5 genes (from disk):
<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 9 stored elements and shape (3, 5)>
  Coords	Values
  (0, 0)	1.0
  (0, 2)	3.0
  (0, 3)	2.0
  (0, 4)	2.0
  (1, 0)	1.0
  (1, 2)	1.0
  (1, 4)	2.0
  (2, 1)	3.0
  (2, 3)	1.0


In [36]:
# obs/var metadata IS in memory and fully usable
# This lets you filter by metadata without loading .X
print("Cell type counts in backed object:")
print(adata_backed.obs["cell_type"].value_counts())

Cell type counts in backed object:
cell_type
Monocyte    193
B cell      165
T cell      142
Name: count, dtype: int64


In [37]:
# Select cells of one type using metadata, then load only that subset
# This is the typical large-data workflow: filter metadata first, then read
b_cell_mask = adata_backed.obs["cell_type"] == "B cell"
b_cells = adata_backed[b_cell_mask].to_memory()  # now load just B cells into RAM

print("B cells loaded into memory:", b_cells)
print("isbacked:", b_cells.isbacked)  # False — now fully in memory

B cells loaded into memory: AnnData object with n_obs × n_vars = 165 × 5000
    obs: 'cell_type'
isbacked: False


In [38]:
# Always close the file handle when done with backed mode
adata_backed.file.close()
print("File handle closed.")

File handle closed.


---
## 9. Views vs Copies — Important Gotcha

A key subtlety when working with AnnData: most subsetting operations return **views** (like NumPy views — no data is copied, modifications are not allowed). When you need to modify the result, call `.copy()` first.

In [39]:
# View: lightweight, no data copied
view = adata[adata.obs["cell_type"] == "Monocyte"]
print("Is view:", view.is_view)  # True

# Trying to add a column to a view raises a warning and implicitly copies
# Best practice: always .copy() before modifying
monocytes = view.copy()
print("Is view after copy:", monocytes.is_view)  # False

# Now safe to modify
monocytes.obs["is_monocyte"] = True
print("Added 'is_monocyte' column:", monocytes.obs["is_monocyte"].all())

Is view: True
Is view after copy: False
Added 'is_monocyte' column: True


---
## 10. Summary

| Component | Attribute | Shape | Purpose |
|---|---|---|---|
| Main matrix | `.X` | (n_obs, n_vars) | Primary expression data |
| Obs metadata | `.obs` | (n_obs, any) | Cell-level annotations |
| Var metadata | `.var` | (n_vars, any) | Gene-level annotations |
| Obs matrices | `.obsm` | (n_obs, k) per key | Cell embeddings (PCA, UMAP) |
| Var matrices | `.varm` | (n_vars, k) per key | Gene loadings |
| Pairwise obs | `.obsp` | (n_obs, n_obs) per key | Cell-cell graphs |
| Unstructured | `.uns` | dict | Colour palettes, parameters |
| Layers | `.layers` | (n_obs, n_vars) per key | Alt. expression matrices |

### Key takeaways

- AnnData keeps all representations of a single dataset in one object — raw counts, normalised values, embeddings, and metadata — without duplication of indices
- `.h5ad` is the standard interchange format for scRNA-seq data in Python (and increasingly in R via `zellkonverter`)
- Backed mode allows working with datasets that are larger than RAM by keeping `.X` on disk
- Subsetting always returns a view; use `.copy()` before modifying

In [40]:
# Clean up output files
import os
for f in ["adata_getting_started.h5ad", "adata_compressed.h5ad", "adata_large.h5ad"]:
    if os.path.exists(f):
        os.remove(f)
print("Cleaned up.")

Cleaned up.
